# Training the LSTM cell with backprop (BPTT)

`lstm_v2` / `lstm_v3` used **hardcoded** gate weights. Here we *learn* them.

We keep the same scalar LSTM cell but add a linear readout and train everything end to end
on the next-number task with **back-propagation through time (BPTT)** and plain gradient
descent. A numerical gradient check confirms the hand-derived gradients are correct.

Because `H` and `C` are scalars and each weight is a 2-vector `[w_h, w_x]`, every gradient
below is a scalar or a 2-vector — the full LSTM math with none of the matrix bookkeeping.

In [1]:
import numpy as np

np.random.seed(1)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

## 1. Forward pass — now with a cache

The gate equations are identical to `lstm_v2`. The only change: we **store every
intermediate value** at each time step, because backprop needs them. We also add a linear
readout `ŷ = w_out · H_T + b_out` on the final hidden state.

Weights are initialised small and random (not zero, or the gates would be symmetric); the
forget-gate bias starts at `+1.0`, a standard trick so the cell remembers by default early on.

In [2]:
class TrainableLSTM:
    def __init__(self):
        r = lambda: np.random.randn(2) * 0.3     # small random [w_h, w_x]
        self.W_f, self.W_i, self.W_c, self.W_o = r(), r(), r(), r()
        self.b_f, self.b_i, self.b_c, self.b_o = 1.0, 0.0, 0.0, 0.0
        self.w_out, self.b_out = np.random.randn() * 0.3, 0.0

    def forward(self, seq):
        """Run the sequence; return prediction, final H, and per-step cache."""
        H, C = 0.0, 0.0
        cache = []
        for x in seq:
            inp = np.array([H, x])
            F = sigmoid(self.W_f @ inp + self.b_f)
            I = sigmoid(self.W_i @ inp + self.b_i)
            C_bar = np.tanh(self.W_c @ inp + self.b_c)
            C_prev = C
            C = F * C_prev + I * C_bar
            O = sigmoid(self.W_o @ inp + self.b_o)
            tanhC = np.tanh(C)
            H = O * tanhC
            cache.append((inp, F, I, C_bar, O, C_prev, tanhC))
        y_hat = self.w_out * H + self.b_out
        return y_hat, H, cache

## 2. Backward pass (BPTT)

Loss is `L = (ŷ − y)²`. We walk the time steps **in reverse**, carrying two gradients:
`dH` (into the hidden state) and `dC` (into the cell state).

The key LSTM detail: `dC` accumulates from **two** paths — the current `H_t` *and* the next
step's cell state flowing back through the forget gate (`dC = dC * F_t` at the end of each
step). Derivatives used: `σ' = s(1−s)` and `tanh' = 1−t²`.

In [3]:
def gradients(model, seq, y):
    """Return (loss, grads dict). grads has one entry per parameter."""
    y_hat, H, cache = model.forward(seq)

    gW = {k: np.zeros(2) for k in ('W_f', 'W_i', 'W_c', 'W_o')}
    gb = {k: 0.0 for k in ('b_f', 'b_i', 'b_c', 'b_o')}

    # readout
    d_yhat = 2 * (y_hat - y)
    g_w_out = d_yhat * H
    g_b_out = d_yhat

    dH = d_yhat * model.w_out    # gradient entering the final hidden state
    dC = 0.0
    for inp, F, I, C_bar, O, C_prev, tanhC in reversed(cache):
        dC = dC + dH * O * (1 - tanhC ** 2)         # merge H-path into cell state
        dO = dH * tanhC

        # gate output-grads -> pre-activation grads
        z_o = dO * O * (1 - O)
        z_f = (dC * C_prev) * F * (1 - F)
        z_i = (dC * C_bar) * I * (1 - I)
        z_c = (dC * I) * (1 - C_bar ** 2)

        # accumulate weight/bias grads (inp = [H_{t-1}, x_t])
        gW['W_f'] += z_f * inp; gb['b_f'] += z_f
        gW['W_i'] += z_i * inp; gb['b_i'] += z_i
        gW['W_c'] += z_c * inp; gb['b_c'] += z_c
        gW['W_o'] += z_o * inp; gb['b_o'] += z_o

        # propagate to previous step (w_h is index 0 of each weight)
        dH = z_f * model.W_f[0] + z_i * model.W_i[0] + z_c * model.W_c[0] + z_o * model.W_o[0]
        dC = dC * F

    grads = {**gW, **gb, 'w_out': g_w_out, 'b_out': g_b_out}
    return (y_hat - y) ** 2, grads

## 3. Gradient check

Before trusting the analytic gradients, compare one against a finite-difference estimate
`(L(w+ε) − L(w−ε)) / 2ε`. They should match to ~6+ digits.

In [4]:
def loss_only(model, seq, y):
    return (model.forward(seq)[0] - y) ** 2

check = TrainableLSTM()
seq, y = np.array([3, 4, 5, 6]) / 50, 7 / 50
_, g = gradients(check, seq, y)

eps = 1e-6
check.W_f[0] += eps; L_plus = loss_only(check, seq, y)
check.W_f[0] -= 2 * eps; L_minus = loss_only(check, seq, y)
check.W_f[0] += eps
numeric = (L_plus - L_minus) / (2 * eps)

print(f"analytic dL/dW_f[0]: {g['W_f'][0]: .6e}")
print(f"numeric  dL/dW_f[0]: {numeric: .6e}")

analytic dL/dW_f[0]: -8.377525e-06
numeric  dL/dW_f[0]: -8.377526e-06


## 4. Train with gradient descent

Same next-number data as `lstm_v3`. Each epoch: average the gradients over all examples
(full-batch) and take a step. Watch the loss fall — the weights are being *learned*, not set.

In [5]:
SCALE, WINDOW, LR, EPOCHS = 50.0, 4, 0.3, 3000
data = [(np.arange(s, s + WINDOW) / SCALE, (s + WINDOW) / SCALE) for s in range(40)]

PARAMS = ('W_f', 'W_i', 'W_c', 'W_o', 'b_f', 'b_i', 'b_c', 'b_o', 'w_out', 'b_out')
model = TrainableLSTM()

for epoch in range(EPOCHS):
    acc = {p: np.zeros_like(getattr(model, p)) for p in PARAMS}
    total = 0.0
    for seq, y in data:
        loss, g = gradients(model, seq, y)
        total += loss
        for p in PARAMS:
            acc[p] = acc[p] + g[p]
    for p in PARAMS:                              # gradient-descent step
        setattr(model, p, getattr(model, p) - LR * acc[p] / len(data))

    if (epoch + 1) % 500 == 0:
        print(f"epoch {epoch + 1:4d} | mean loss {total / len(data):.6f}")

epoch  500 | mean loss 0.000117
epoch 1000 | mean loss 0.000102
epoch 1500 | mean loss 0.000090
epoch 2000 | mean loss 0.000081
epoch 2500 | mean loss 0.000073
epoch 3000 | mean loss 0.000067


## 5. Predict with the learned weights

In [6]:
def predict_next(seq):
    return model.forward(np.asarray(seq) / SCALE)[0] * SCALE

for seq in ([2, 3, 4, 5], [10, 11, 12, 13], [22, 23, 24, 25], [35, 36, 37, 38]):
    print(f"{seq} -> {predict_next(seq):5.2f}  (actual {seq[-1] + 1})")

[2, 3, 4, 5] ->  6.46  (actual 6)
[10, 11, 12, 13] -> 13.53  (actual 14)
[22, 23, 24, 25] -> 26.21  (actual 26)
[35, 36, 37, 38] -> 38.92  (actual 39)


The predictions are learned end to end. They land within a fraction of the true value —
impressive for a cell whose entire memory is **one scalar**. To sharpen them you'd give
`H`/`C` more dimensions (making the weights matrices, with `gW += np.outer(z_gate, inp)`),
add mini-batching, or swap plain SGD for Adam — but the BPTT math above stays exactly the same.